# U1-05 · Un recomendador de películas con MovieLens

*Programación de Redes Neuronales - Machine Learning (IFCD48) · Unidad 1, apartado 1.8*

Ejecuta cada celda con **Mayús + Intro** (en Colab, también con el botón ▶). La primera celda descarga los datos y entrena la factorización: tarda alrededor de un minuto.

In [ ]:
import os, urllib.request, zipfile
import numpy as np
import pandas as pd

url = "https://files.grouplens.org/datasets/movielens/ml-100k.zip"
if not os.path.exists("ml-100k.zip"):
    urllib.request.urlretrieve(url, "ml-100k.zip")
zipfile.ZipFile("ml-100k.zip").extractall()
notas = pd.read_csv("ml-100k/u.data", sep="\t",
                    names=["usuario", "peli", "nota", "fecha"])
titulos = pd.read_csv("ml-100k/u.item", sep="|", encoding="latin-1",
                      header=None, usecols=[0, 1], index_col=0)[1]
print(len(notas), "notas de", notas.usuario.nunique(), "usuarios a",
      notas.peli.nunique(), "películas")

rng = np.random.default_rng(0)
prueba = rng.random(len(notas)) < 0.2       # 20 % para evaluar
ent, pru = notas[~prueba], notas[prueba]
u, p, r = (ent[c].to_numpy() - (c != "nota")
           for c in ("usuario", "peli", "nota"))     # índices desde 0
up, pp, rp = (pru[c].to_numpy() - (c != "nota")
              for c in ("usuario", "peli", "nota"))
rmse = lambda pred: np.sqrt(np.mean((pred - rp) ** 2))
media = r.mean()
base = rmse(np.full(len(rp), media))
print(f"predecir siempre la media: RMSE {base:.3f}")

# factorización de matrices: nota ≈ media + sesgos + p_u · q_i
def entrenar(k, epocas=30, eta=0.01, lam=0.05):
    P = rng.normal(0, 0.1, (943, k))
    Q = rng.normal(0, 0.1, (1682, k))
    bu, bi = np.zeros(943), np.zeros(1682)
    for epoca in range(epocas):
        for j in rng.permutation(len(r)):        # SGD, nota a nota
            a, b = u[j], p[j]
            e = r[j] - (media + bu[a] + bi[b] + P[a] @ Q[b])
            bu[a] += eta * (e - lam * bu[a])
            bi[b] += eta * (e - lam * bi[b])
            P[a], Q[b] = (P[a] + eta * (e * Q[b] - lam * P[a]),
                          Q[b] + eta * (e * P[a] - lam * Q[b]))
    return lambda us, ps: (media + bu[us] + bi[ps]
                           + np.sum(P[us] * Q[ps], axis=1))

for k in (0, 20):            # k = 0: solo sesgos
    modelo = entrenar(k)
    print(f"k = {k:2d}: RMSE {rmse(modelo(up, pp)):.3f}")

# recomendaciones para el usuario 1 (índice 0), sin repetir vistas
vistas = set(p[u == 0])
nuevas = np.array([i for i in range(1682) if i not in vistas])
nota_est = modelo(np.zeros(len(nuevas), int), nuevas)
for i in nuevas[np.argsort(nota_est)[::-1][:5]]:
    print(f"  {titulos[i + 1]}")

**Prueba.** Cambia `k` a 50 y `epocas` a 60. ¿Mejora el error en prueba o empieza a empeorar?

## Recomendación basada en contenido

In [ ]:
import numpy as np
import pandas as pd

generos = ["desconocido", "acción", "aventura", "animación",
           "infantil", "comedia", "crimen", "documental", "drama",
           "fantasía", "cine negro", "terror", "musical", "misterio",
           "romance", "ciencia ficción", "suspense", "bélico",
           "western"]
pelis = pd.read_csv("ml-100k/u.item", sep="|", encoding="latin-1",
                    header=None, index_col=0)
titulo, G = pelis[1], pelis.iloc[:, 4:].to_numpy(float)
print("películas:", G.shape[0], "| géneros por película, media:",
      G.sum(axis=1).mean().round(2))

def parecidas(i, M, n=5):               # similitud del coseno
    norma = np.linalg.norm(M, axis=1) + 1e-12
    sim = (M @ M[i - 1]) / (norma * norma[i - 1])
    sim[i - 1] = -1                      # quita la propia
    mejores = np.argsort(sim)[::-1][:n]
    return [(titulo[j + 1], round(sim[j], 2)) for j in mejores]

print([generos[g] for g in np.flatnonzero(G[49])])
for t, s in parecidas(50, G):           # 50 = Star Wars (1977)
    print(f"  {s}  {t}")

## Filtrado colaborativo por vecinos

In [ ]:
import numpy as np
import pandas as pd

notas = pd.read_csv("ml-100k/u.data", sep="\t",
                    names=["usuario", "peli", "nota", "fecha"])
titulo = pd.read_csv("ml-100k/u.item", sep="|", encoding="latin-1",
                     header=None, index_col=0)[1]
R = notas.pivot(index="usuario", columns="peli", values="nota")
C = R.sub(R.mean(axis=1), axis=0).fillna(0).to_numpy()  # centrada
vistas = R.notna().to_numpy().sum(axis=0)

norma = np.linalg.norm(C, axis=0) + 1e-12
S = (C.T @ C) / np.outer(norma, norma)    # coseno entre películas
np.fill_diagonal(S, -1)
S[:, vistas < 50] = -1                    # al menos 50 valoraciones
for j in np.argsort(S[49])[::-1][:5]:     # Star Wars (1977)
    print(f"  {S[49, j]:.2f}  {titulo[R.columns[j]]}")

## Evaluar una lista de recomendaciones

In [ ]:
import numpy as np
import pandas as pd

notas = pd.read_csv("ml-100k/u.data", sep="\t",
                    names=["usuario", "peli", "nota", "fecha"])
rng = np.random.default_rng(0)
prueba = rng.random(len(notas)) < 0.2       # el mismo reparto
ent, pru = notas[~prueba], notas[prueba]
R = np.zeros((943, 1682))
R[ent.usuario - 1, ent.peli - 1] = ent.nota
vista = R > 0

def precision_10(puntos):                # puntos: usuarios x pelis
    puntos = np.where(vista, -np.inf, puntos)  # no repetir vistas
    top = np.argsort(-puntos, axis=1)[:, :10]
    gusta = pru[pru.nota >= 4]
    rel = set(zip(gusta.usuario - 1, gusta.peli - 1))
    usuarios = {u for u, _ in rel}
    aciertos = [sum((u, i) in rel for i in top[u]) for u in usuarios]
    return np.mean(aciertos) / 10

popular = np.tile(vista.sum(axis=0), (943, 1)).astype(float)
C = np.where(vista, R - R.sum(1, keepdims=True)
             / vista.sum(1, keepdims=True), 0)
n = np.linalg.norm(C, axis=0) + 1e-12
S = (C.T @ C) / np.outer(n, n); np.fill_diagonal(S, 0)
vecinos = C @ S                    # películas parecidas a las suyas
for nombre, p in (("al azar", rng.random((943, 1682))),
                  ("las más vistas", popular),
                  ("vecinos por película", vecinos)):
    print(f"{nombre:<21} precisión@10 = {precision_10(p):.3f}")

**Ejercicio.** Mezcla las dos puntuaciones, `0.5 * popular / popular.max() + 0.5 * vecinos / vecinos.max()`, y mide su precisión@10.

> Datos: MovieLens 100K, GroupLens Research, Universidad de Minnesota (Harper y Konstan, 2015, https://doi.org/10.1145/2827872). Se descargan de su web; no se pueden redistribuir.
>
> Koren, Y., Bell, R. y Volinsky, C. (2009). Matrix factorization techniques for recommender systems. *Computer*, 42(8), 30–37. https://doi.org/10.1109/MC.2009.263
>
> Cremonesi, P., Koren, Y. y Turrin, R. (2010). Performance of recommender algorithms on top-N recommendation tasks. *RecSys '10*, 39–46. https://doi.org/10.1145/1864708.1864721